In [11]:
import pandas as pd

src = "C:/Users/SHAH KAVYA RONAK/Downloads/Dataset for Data Analytics.xlsx"
df = pd.read_excel(src)

log = []

# 1. Missing values audit
nulls_before = df.isnull().sum()
df['CouponCode'] = df['CouponCode'].fillna('No Coupon')
log.append(['CR001', "Imputed missing CouponCode with 'No Coupon'", f"{nulls_before['CouponCode']} records updated", 'Resolved'])

# 2. Duplicate audit
dupe_rows = df.duplicated().sum()
dupe_ids = df['OrderID'].duplicated().sum()
log.append(['CR002', 'Checked for fully duplicate rows', f'{dupe_rows} found', 'Verified'])
log.append(['CR003', 'Checked for duplicate OrderIDs', f'{dupe_ids} found', 'Verified'])
df = df.drop_duplicates()

# 3. Text standardization
text_cols = ['Product','ShippingAddress','PaymentMethod','OrderStatus','ReferralSource','CustomerID','OrderID','TrackingNumber','CouponCode']
changed_text = 0
for c in text_cols:
    if c in df.columns:
    
        before = df[c].copy()
        df[c] = df[c].astype(str).str.strip()
        changed_text += (before.astype(str) != df[c]).sum()
log.append(['CR004', 'Trimmed whitespace across text columns', f'{changed_text} cells affected', 'Resolved'])

# 4. Date format standardization
df['Date'] = pd.to_datetime(df['Date'],errors='coerce').dt.strftime('%Y-%m-%d')
log.append(['CR005', 'Standardized Date column to ISO 8601 (YYYY-MM-DD)', f'{len(df)} records formatted', 'Resolved'])

# 5. Numeric precision
df['UnitPrice'] = df['UnitPrice'].round(2)
df['TotalPrice'] = df['TotalPrice'].round(2)
log.append(['CR006', 'Enforced 2-decimal precision on UnitPrice and TotalPrice', f'{len(df)} records checked', 'Resolved'])

# 6. Cross-field validation: Quantity x UnitPrice = TotalPrice
calc = (df['Quantity'] * df['UnitPrice']).round(2)
mismatches = (calc - df['TotalPrice']).abs() > 0.01
log.append(['CR007', 'Validated TotalPrice = Quantity x UnitPrice', f'{mismatches.sum()} mismatches found', 'Verified'])

# 7. Final integrity gate checks
final_dupe_ids = df['OrderID'].duplicated().sum()
bad_dates = pd.to_datetime(df['Date'], format='%Y-%m-%d', errors='coerce').isna().sum()
log.append(['CR008', 'Verification gate: 0% error rate on unique identifiers and date formats', f'{final_dupe_ids} duplicate IDs, {bad_dates} bad dates', 'Passed'])

out_xlsx = r"C:/Users/SHAH KAVYA RONAK/Downloads/Cleaned_Dataset.xlsx"
df.to_excel(out_xlsx, index=False)

log_df = pd.DataFrame(log, columns=['Change ID','Description','Impact','Status'])
log_df.to_csv(r"C:/Users/SHAH KAVYA RONAK/Downloads/Change_Log_Report.csv", index=False)
print(log_df.to_string(index=False))
print('\nFinal shape:', df.shape)

Change ID                                                             Description                       Impact   Status
    CR001                             Imputed missing CouponCode with 'No Coupon'          309 records updated Resolved
    CR002                                        Checked for fully duplicate rows                      0 found Verified
    CR003                                          Checked for duplicate OrderIDs                      0 found Verified
    CR004                                  Trimmed whitespace across text columns             0 cells affected Resolved
    CR005                       Standardized Date column to ISO 8601 (YYYY-MM-DD)       1200 records formatted Resolved
    CR006                Enforced 2-decimal precision on UnitPrice and TotalPrice         1200 records checked Resolved
    CR007                             Validated TotalPrice = Quantity x UnitPrice           0 mismatches found Verified
    CR008 Verification gate: 0% error ra